# Member A 使用仓库公共协议

先认识 patient 患者、visit 访视、origin 预测起点、horizon 预测跨度、target 单项目标、OOF 折外预测。所有实现与文字有 Codex 协助，先阅读 AI_USAGE.md。

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'common').exists():
    ROOT = ROOT.parent
assert (ROOT / 'common').exists(), '请在仓库根目录或 member_a 中运行'
sys.path.insert(0, str(ROOT))
from common.metric import evaluate_oof, smape_plus_one
shared = ROOT / 'artifacts/shared'
manifest = json.loads((shared / 'manifest.json').read_text())
samples = pd.read_csv(shared / 'samples.csv')
patients = pd.read_csv(shared / 'patient_folds.csv')
print('协议', manifest['protocol'], 'seed', manifest['seed'])
print('患者', len(patients), '标量预测请求', len(samples))

协议 patient5-omics-smape1-v1 seed 4486
患者 248 标量预测请求 12697


## 1 一行样本是什么

当前 samples.csv 一行是一个 prediction_id 对应的单项 UPDRS 评分，不是一行四个目标。不能把旧本地宽表直接拼入。

In [2]:
print(samples.head(8).to_string(index=False))

visit_id  patient_id  visit_month  horizon  target  target_month                  prediction_id  rating  fold
 10053_0       10053            0        0 updrs_1             0  10053_0_updrs_1_plus_0_months     3.0     3
 10053_0       10053            0       12 updrs_1            12 10053_0_updrs_1_plus_12_months     4.0     3
 10053_0       10053            0       24 updrs_1            24 10053_0_updrs_1_plus_24_months     4.0     3
 10053_0       10053            0        6 updrs_1             6  10053_0_updrs_1_plus_6_months     2.0     3
 10053_0       10053            0        0 updrs_2             0  10053_0_updrs_2_plus_0_months     0.0     3
 10053_0       10053            0       12 updrs_2            12 10053_0_updrs_2_plus_12_months     2.0     3
 10053_0       10053            0       24 updrs_2            24 10053_0_updrs_2_plus_24_months     3.0     3
 10053_0       10053            0        6 updrs_2             6  10053_0_updrs_2_plus_6_months     4.0     3


## 2 为什么按患者分折

同一患者所有访视跟随同一个 fold。不能随机拆分请求行。

In [3]:
for fold in range(5):
    train = set(samples.loc[samples.fold != fold, 'patient_id'])
    valid = set(samples.loc[samples.fold == fold, 'patient_id'])
    assert train.isdisjoint(valid)
    print(f'fold {fold}: 训练 {len(train)} 位患者，验证 {len(valid)} 位患者')

fold 0: 训练 198 位患者，验证 50 位患者
fold 1: 训练 198 位患者，验证 50 位患者
fold 2: 训练 198 位患者，验证 50 位患者
fold 3: 训练 199 位患者，验证 49 位患者
fold 4: 训练 199 位患者，验证 49 位患者


## 3 手算 SMAPE+1

不是最后分数加 1，而是给真实值和预测值先各加 1。

In [4]:
print('真实 [0,2]、预测 [0,4] 的分数 =', smape_plus_one([0,2], [0,4]))

真实 [0,2]、预测 [0,4] 的分数 = 25.0


## 4 重算所有 OOF

mean_score 是五折等权均分；oof_score 是所有请求合并评分，不能混淆。

In [5]:
for name in ['A0','A1','A2','A3']:
    predictions = pd.read_csv(ROOT / 'member_a/results' / name / 'oof.csv')
    result = evaluate_oof(samples, predictions)
    print(name, 'mean_score =', round(result['mean_score'],6),
          'oof_score =', round(result['oof_score'],6), '请求数 =', result['n_predictions'])

A0 mean_score = 65.62066 oof_score = 65.484091 请求数 = 12697
A1 mean_score = 66.125865 oof_score = 65.990057 请求数 = 12697
A2 mean_score = 66.753731 oof_score = 66.631727 请求数 = 12697
A3 mean_score = 66.815759 oof_score = 66.649858 请求数 = 12697


![目标分布](eda/figures/02_targets.png)

![特征实验](eda/figures/04_ablation.png)

所有扩展在当前模型上没有改善五折均分。查看 features.py 理解时间限制，查看 run.py 理解公共接口的调用。完整重训命令在 README.md。